## What is Text Chunking?
- Chunking is splitting a large document into smaller pieces (chunks) that can be embedded and retrieved individually. The goal is to make each chunk   semantically meaningful and small enough to fit in a retrieval context.

## Why Does Chunking Matter?

| Problem | Effect on RAG |
|---------|---------------|
| Chunks too large | Noisy retrieval, exceeds context window| 
| Chunks too small | Lose context, fragments meaning |
| Bad boundaries | Cuts sentences mid-thought |

In [2]:
# 1. Fixed-Size Chunking
# Split by a fixed number of characters or words, regardless of content.
# ✅ Simple | ❌ May cut mid-sentence

def fixed_chunk_size(text, chunk_size=200):
    chunks = []
    for i in range(0,len(text),chunk_size):   # How range() works here range(start, stop, step) generates a sequence of numbers:
        chunks.append(text[i:i + chunk_size])  # slicing usinfg [i:i+chunk_size]
    return chunks

text = "Chunking is splitting a large document into smaller pieces (chunks) that can be embedded and retrieved individually. The goal is to make each chunk   semantically meaningful and small enough to fit in a retrieval context."
chunked_text = fixed_chunk_size(text)
print(chunked_text)

['Chunking is splitting a large document into smaller pieces (chunks) that can be embedded and retrieved individually. The goal is to make each chunk   semantically meaningful and small enough to fit in', ' a retrieval context.']


In [4]:
# 2. Fixed-Size with Overlap
# Same as above but each chunk overlaps with the previous one to preserve context across boundaries.
# ✅ Preserves cross-boundary context | ❌ Slightly redundant data

def chunk_with_overlap(text,chunk_size = 200, overlap = 50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

text = "Chunking is splitting a large document into smaller pieces (chunks) that can be embedded and retrieved individually. The goal is to make each chunk   semantically meaningful and small enough to fit in a retrieval context."
chunked_text = chunk_with_overlap(text)
print(chunked_text)

['Chunking is splitting a large document into smaller pieces (chunks) that can be embedded and retrieved individually. The goal is to make each chunk   semantically meaningful and small enough to fit in', 'semantically meaningful and small enough to fit in a retrieval context.']


In [16]:
# leaving strip() empty will remove extra whit spaces only in the beginning or ending of Strin. but extra whit spaces inside the string will not be removed
text = '    Every        Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.'

print(text.strip())

Every        Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.


In [15]:
# to remove extra spaces inside the string we can do the following

text = '    Every        Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.'
print(text.split())
cleaned_text = ' '.join(text.split())
print(cleaned_text)

['Every', 'Friday,', 'we', 'have', 'a', 'standup', 'meeting.', 'The', 'only', 'reason', 'why', 'we', 'might', 'not', 'have', 'a', 'meeting', 'on', 'a', 'Friday', 'is', 'public', 'holiday.', 'That', 'Friday,', 'we', 'talk', 'about', 'what', 'we', 'did', 'in', 'the', 'previous', 'week,', 'and', 'what', 'we', 'are', 'going', 'to', 'do', 'in', 'the', 'week', 'starting', 'from', 'that', 'Friday.']
Every Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.


In [8]:
# (?<=...)   lookbehind  ←  "what's behind me?"
# (?=...)    lookahead   →  "what's ahead of me?"

import re

text = 'Every Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.'

result = re.split(r'(?<=[.!?])\s+', text.strip()) # Typical use case: '(?<=[.!?])\s+' This is a classic sentence boundary splitter. You'd use it with re.split() to break a paragraph into individual sentences:
print(text.strip())
print(result)

Every Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.
['Every Friday, we have a standup meeting.', 'The only reason why we might not have a meeting on a Friday is public holiday.', 'That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.']


In [24]:
# 3. Sentence-Based Chunking
# Use regex (which you already know!) to split on sentence boundaries.
# ✅ Respects natural language boundaries | ❌ Sentences can vary wildly in lengt

import re

def sentence_chunk(text, max_sentences = 3):
    sentence = re.split(r'(?<=[.!?])\s+', text.strip())
    print(sentence)
    print(len(sentence))
    chunks = []
    for i in range(0,len(sentence),max_sentences):
        chunk = ' '.join(sentence[i:i+max_sentences])
        chunks.append(chunk)
    return chunks

text = 'Every Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday. And then we enjoy weekend'

result = sentence_chunk(text)
print(result)

['Every Friday, we have a standup meeting.', 'The only reason why we might not have a meeting on a Friday is public holiday.', 'That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.', 'And then we enjoy weekend']
4
['Every Friday, we have a standup meeting. The only reason why we might not have a meeting on a Friday is public holiday. That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday.', 'And then we enjoy weekend']


In [31]:
# 4. Paragraph-Based Chunking
# Split on blank lines — great for structured documents like markdown or articles.
# ✅ Very natural boundaries | ❌ Paragraphs can be too long or too short

def paragraph_cunk(text):
    paragraphs = re.split(r'\n\s*\n' , text.strip())  # Find any place where two newlines are separated by nothing (or only whitespace) — which is exactly what a blank line looks like.
    
    chunks = [p.strip() for p in paragraphs if p.strip()]
    return chunks

text = '''Every Friday, we have a standup meeting.

The only reason why we might not have a meeting on a Friday is public holiday. 

That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday. And then we enjoy weekend'''
paragraphs_result = paragraph_cunk(text)
print(paragraphs_result)


['Every Friday, we have a standup meeting.', 'The only reason why we might not have a meeting on a Friday is public holiday.', 'That Friday, we talk about what we did in the previous week, and what we are going to do in the week starting from that Friday. And then we enjoy weekend']
